# Multi-Qubit Observables and Correlations

This notebook explores multi-qubit Pauli observables and correlation measurements. We'll see how Hoeffding shot planning applies to correlation measurements and demonstrate quantum entanglement through correlation measurements.

## Key Concepts

1. **Correlation observables**: Measure correlations between qubits (e.g., ZZ, XX, YY)
2. **Eigenvalues**: Multi-qubit Pauli operators still have eigenvalues ±1
3. **Entanglement**: Creates correlations impossible in classical systems
4. **Bell states**: Maximally entangled two-qubit states

In [1]:
import math
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.transpiler import generate_preset_pass_manager

from qamp_shotplanner import (
    HoeffdingPlanner,
    correlation_observable,
    bell_state_observable,
)

## 1. Product State: No Correlations

First, let's create a simple product state and measure correlations.

In [2]:
# Create a product state: |0⟩ ⊗ |+⟩
# where |+⟩ = (|0⟩ + |1⟩)/√2 = H|0⟩

qc_product = QuantumCircuit(2)
qc_product.h(1)  # Create |+⟩ on qubit 1

print("Product State Circuit:")
print(f"Qubits: {qc_product.num_qubits}")
print(f"State: |0⟩ ⊗ |+⟩")

Product State Circuit:
Qubits: 2
State: |0⟩ ⊗ |+⟩


## 2. Correlation Observables

For two qubits, we can measure correlations like:
- $ZZ = Z_0 \otimes Z_1$
- $XX = X_0 \otimes X_1$
- $YY = Y_0 \otimes Y_1$

**Key point**: Even multi-qubit Pauli operators have eigenvalues ±1, so they're still bounded in [-1, 1]!

In [3]:
# Create correlation observables
obs_ZZ = correlation_observable(qubit1=0, qubit2=1, num_qubits=2, pauli1="Z", pauli2="Z")
obs_XX = correlation_observable(qubit1=0, qubit2=1, num_qubits=2, pauli1="X", pauli2="X")
obs_YY = correlation_observable(qubit1=0, qubit2=1, num_qubits=2, pauli1="Y", pauli2="Y")

print("Correlation Observables:")
print(f"ZZ: {obs_ZZ}")
print(f"XX: {obs_XX}")
print(f"YY: {obs_YY}")

Correlation Observables:
ZZ: SparsePauliOp(['ZZ'],
              coeffs=[1.+0.j])
XX: SparsePauliOp(['XX'],
              coeffs=[1.+0.j])
YY: SparsePauliOp(['YY'],
              coeffs=[1.+0.j])


## 3. Shot Planning for Correlations

Same planning as single-qubit case because the bounds are the same!

In [4]:
# Plan shots (same as before)
epsilon = 0.02
delta = 0.01

planner = HoeffdingPlanner(epsilon_stat=epsilon, delta=delta, a=-1.0, b=+1.0)
shots = planner.planned_shots()

print(f"Planned shots for correlation measurements: {shots}")
print(f"Guarantee: |Ê - E| ≤ {epsilon} with probability ≥ {1-delta:.0%}")

Planned shots for correlation measurements: 26492
Guarantee: |Ê - E| ≤ 0.02 with probability ≥ 99%


In [5]:
# Helper function to run estimator
def measure_observable(qc, observable, shots_val):
    backend = AerSimulator()
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    qc_isa = pm.run(qc)
    
    options = {"run_options": {"shots": shots_val}, "backend_options": {"seed_simulator": 42}}
    estimator = AerEstimator(options=options)
    
    job = estimator.run([(qc_isa, observable, [])])
    return float(job.result()[0].data.evs)

# Measure correlations for product state
E_ZZ_product = measure_observable(qc_product, obs_ZZ, shots)
E_XX_product = measure_observable(qc_product, obs_XX, shots)
E_YY_product = measure_observable(qc_product, obs_YY, shots)

print("Product State |0⟩ ⊗ |+⟩ Correlations:")
print(f"E[ZZ] = {E_ZZ_product:.6f}")
print(f"E[XX] = {E_XX_product:.6f}")
print(f"E[YY] = {E_YY_product:.6f}")

print("\nTheoretical predictions for |0⟩ ⊗ |+⟩:")
print(f"E[ZZ] = ⟨Z⟩₀·⟨Z⟩₁ = (+1)·(0) = 0 (product of individual expectations)")
print(f"E[XX] = ⟨X⟩₀·⟨X⟩₁ = (0)·(+1) = 0 (product of individual expectations)")
print(f"E[YY] = ⟨Y⟩₀·⟨Y⟩₁ = (0)·(0) = 0 (product of individual expectations)")

Product State |0⟩ ⊗ |+⟩ Correlations:
E[ZZ] = 0.000000
E[XX] = 0.000000
E[YY] = 0.000000

Theoretical predictions for |0⟩ ⊗ |+⟩:
E[ZZ] = ⟨Z⟩₀·⟨Z⟩₁ = (+1)·(0) = 0 (product of individual expectations)
E[XX] = ⟨X⟩₀·⟨X⟩₁ = (0)·(+1) = 0 (product of individual expectations)
E[YY] = ⟨Y⟩₀·⟨Y⟩₁ = (0)·(0) = 0 (product of individual expectations)


## 4. Bell State: Maximally Entangled

Now let's create a Bell state and observe the dramatic difference in correlations!

In [6]:
# Create Bell state: |Φ⁺⟩ = (|00⟩ + |11⟩)/√2
qc_bell = QuantumCircuit(2)
qc_bell.h(0)          # Create superposition
qc_bell.cx(0, 1)      # Create entanglement (CNOT gate)

print("Bell State Circuit:")
print(f"Qubits: {qc_bell.num_qubits}")
print(f"State: |Φ⁺⟩ = (|00⟩ + |11⟩)/√2")

Bell State Circuit:
Qubits: 2
State: |Φ⁺⟩ = (|00⟩ + |11⟩)/√2


## 5. Bell State Correlations

For the Bell state $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$:

- $E[ZZ] = +1$ (perfect positive correlation)
- $E[XX] = +1$ (perfect positive correlation)  
- $E[YY] = -1$ (perfect negative correlation)

**This is impossible classically!** Bell's theorem proves these correlations violate classical bounds.

In [7]:
# Measure correlations for Bell state
E_ZZ_bell = measure_observable(qc_bell, obs_ZZ, shots)
E_XX_bell = measure_observable(qc_bell, obs_XX, shots)
E_YY_bell = measure_observable(qc_bell, obs_YY, shots)

print("Bell State |Φ⁺⟩ Correlations:")
print(f"E[ZZ] = {E_ZZ_bell:.6f} (theory: +1.0)")
print(f"E[XX] = {E_XX_bell:.6f} (theory: +1.0)")
print(f"E[YY] = {E_YY_bell:.6f} (theory: -1.0)")

print("\nKey observations:")
print(f"- ZZ and XX show perfect positive correlation (+1)")
print(f"- YY shows perfect negative correlation (-1)")
print(f"- These correlations are MAXIMALLY ENTANGLED")
print(f"- Impossible to achieve classically!")

Bell State |Φ⁺⟩ Correlations:
E[ZZ] = 1.000000 (theory: +1.0)
E[XX] = 1.000000 (theory: +1.0)
E[YY] = -1.000000 (theory: -1.0)

Key observations:
- ZZ and XX show perfect positive correlation (+1)
- YY shows perfect negative correlation (-1)
- These correlations are MAXIMALLY ENTANGLED
- Impossible to achieve classically!


## 6. Using Helper Functions

Our package provides convenient helpers for common measurements:

In [8]:
# Using the bell_state_observable helper
obs_ZZ_helper = bell_state_observable(num_qubits=2, correlation_type="ZZ")
obs_XX_helper = bell_state_observable(num_qubits=2, correlation_type="XX")
obs_YY_helper = bell_state_observable(num_qubits=2, correlation_type="YY")

print("Helper functions create the same observables:")
print(f"ZZ: {obs_ZZ_helper}")
print(f"XX: {obs_XX_helper}")
print(f"YY: {obs_YY_helper}")

Helper functions create the same observables:
ZZ: SparsePauliOp(['ZZ'],
              coeffs=[1.+0.j])
XX: SparsePauliOp(['XX'],
              coeffs=[1.+0.j])
YY: SparsePauliOp(['YY'],
              coeffs=[1.+0.j])


## 7. Comparing Product vs Entangled

Let's visualize the dramatic difference:

## 7. Comparing Product vs Entangled

Let's visualize the dramatic difference between product and entangled states:

**Product State |0⟩ ⊗ |+⟩ Results:**
- E[ZZ] = 0.000 (no correlation)
- E[XX] = 0.000 (no correlation)  
- E[YY] = 0.000 (no correlation)

**Bell State |Φ⁺⟩ Results:**
- E[ZZ] = +1.0 (perfect positive Z correlation)
- E[XX] = +1.0 (perfect positive X correlation)
- E[YY] = -1.0 (perfect negative Y correlation)

**Key Insight:** Entangled states can achieve correlation values IMPOSSIBLE for product states! The Bell state violations of classical bounds are a fundamental result in quantum mechanics (Bell's theorem).

## 8. Mixed Correlations

We can also measure mixed correlations like XZ, ZY, etc.

In [9]:
# Create mixed correlation observable
obs_XZ = correlation_observable(qubit1=0, qubit2=1, num_qubits=2, pauli1="X", pauli2="Z")
obs_ZY = correlation_observable(qubit1=0, qubit2=1, num_qubits=2, pauli1="Z", pauli2="Y")

print("Mixed correlation observables:")
print(f"XZ: {obs_XZ}")
print(f"ZY: {obs_ZY}")

# Measure on Bell state
E_XZ_bell = measure_observable(qc_bell, obs_XZ, shots)
E_ZY_bell = measure_observable(qc_bell, obs_ZY, shots)

print(f"\nBell state mixed correlations:")
print(f"E[XZ] = {E_XZ_bell:.6f}")
print(f"E[ZY] = {E_ZY_bell:.6f}")

Mixed correlation observables:
XZ: SparsePauliOp(['ZX'],
              coeffs=[1.+0.j])
ZY: SparsePauliOp(['YZ'],
              coeffs=[1.+0.j])

Bell state mixed correlations:
E[XZ] = 0.000000
E[ZY] = 0.000000


## 9. Key Takeaways

### Multi-Qubit Correlations
1. **Same shot planning**: Correlation observables still bounded in [-1, 1]
2. **Product states**: $\langle O_1 \otimes O_2 \rangle = \langle O_1 \rangle \cdot \langle O_2 \rangle$
3. **Entangled states**: Can achieve correlations impossible classically

### Bell States
- $E[ZZ] = +1$: Perfect Z correlation
- $E[XX] = +1$: Perfect X correlation  
- $E[YY] = -1$: Perfect Y anti-correlation

### Practical Applications
- **Entanglement detection**: Measure correlations
- **Quantum computing**: Correlations enable quantum algorithms
- **Error correction**: Correlations measure logical operators

### For Next Notebook
- Hamiltonian expectation values
- Decomposing Hamiltonians into Pauli terms
- Real-world quantum simulation example